# IBM India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** ibm.com/careers/search

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_CODE   = ""  # e.g. "in" for SmartRecruiters country= param; "" = all
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-03-31 23:37:19
Location filter: '' (empty = broad/global scraping)


In [3]:
COMPANY = "IBM_India"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/IBM_India/Outputs/2026_03_31


In [4]:
print("=" * 60)
print("IBM INDIA JOB SCRAPER")
print("Source: Google Cloud Talent Solution API (jobsapi-google.m-cloud.io)")
print("=" * 60)

ibm_jobs = []
session = get_session()
session.headers.update({
    "Accept": "application/json",
    "Content-Type": "application/json",
})

# IBM uses Google Cloud Talent Solution API
api_url = "https://jobsapi-google.m-cloud.io/api/job/search"
ibm_company_id = "companies/728ae96b-0028-4d31-9697-9b42f37dd3f4"

print("  Using Google Cloud Talent Solution API...")
page_token = ""
page_num = 0

while len(ibm_jobs) < 500:
    payload = {
        "companyName": ibm_company_id,
        "pageSize": 20,
        "offset": page_num * 20,
        "searchText": "",
        "locationFilters": [{"address": "India", "distanceInMiles": 0}],
        "customAttributeFilter": "",
    }
    if page_token:
        payload["pageToken"] = page_token

    try:
        resp = session.post(api_url, json=payload, timeout=30)
        if resp.status_code != 200:
            print(f"  [ERROR] HTTP {resp.status_code}")
            # Try alternate endpoint format
            if page_num == 0:
                print("  Trying alternate GET endpoint...")
                alt_url = f"{api_url}?companyName={ibm_company_id}&pageSize=20&location=India"
                resp = session.get(alt_url, timeout=30)
                if resp.status_code != 200:
                    print(f"  Alt also returned HTTP {resp.status_code}")
                    break
            else:
                break

        data = resp.json()
        matched_jobs = data.get("matchingJobs", data.get("jobs", []))
        total = data.get("totalSize", data.get("total", 0))
        page_token = data.get("nextPageToken", "")

        if page_num == 0:
            print(f"  Total matching jobs: {total}")

        if not matched_jobs:
            break

        print(f"  Page {page_num+1}: {len(matched_jobs)} jobs")

        for mj in matched_jobs:
            job = mj.get("job", mj)
            title = job.get("title", job.get("name", ""))
            desc = job.get("description", "")
            locations = job.get("locations", [])
            city = locations[0].split(",")[0].strip() if locations else "India"
            custom = job.get("customAttributes", {})
            category = custom.get("primary_category", {}).get("stringValues", [""])[0] if "primary_category" in custom else ""
            req_id = job.get("requisitionId", job.get("name", "").split("/")[-1])
            posted = job.get("postingPublishTime", job.get("postingCreateTime", ""))

            if title and is_valid_job_title(title):
                ibm_jobs.append({
                    "job_id": str(req_id),
                    "title": title,
                    "company_name": "IBM",
                    "raw_jd_text": html_to_text(desc),
                    "location_city": city,
                    "industry": "Technology / IT Services",
                    "date_posted": posted[:10] if posted else datetime.now().strftime("%Y-%m-%d"),
                    "is_active": True,
                    "job_url": f"https://www.ibm.com/careers/job/{req_id}",
                    "business_unit": category,
                    "source_platform": "IBM Google CTS API",
                })

        if not page_token:
            break
        page_num += 1
        time.sleep(random.uniform(0.5, 1.5))

    except Exception as e:
        print(f"  [ERROR] {e}")
        break

# Selenium fallback if API fails
if len(ibm_jobs) < 5:
    print("\n  API approach returned few results. Trying Selenium on ibm.com/careers...")
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC

    driver = setup_selenium()
    try:
        driver.get("https://www.ibm.com/careers/search?field_keyword_18[0]=India")
        time.sleep(12)

        # Wait for Angular/React rendering
        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*=\'/job/\']"))
            )
        except:
            time.sleep(5)

        soup = BeautifulSoup(driver.page_source, "lxml")
        # Target actual job links, not UI buttons
        job_links = soup.select("a[href*=\'/careers/job/\'], a[href*=\'/job/\']")
        for link in job_links:
            title = link.get_text(strip=True)
            href = link.get("href", "")
            if is_valid_job_title(title) and title not in [j["title"] for j in ibm_jobs]:
                card = link.parent
                loc_el = card.select_one("[class*=\'location\']") if card else None
                loc = loc_el.get_text(strip=True) if loc_el else "India"
                ibm_jobs.append({
                    "job_id": href.split("/")[-1] if href else str(len(ibm_jobs)),
                    "title": title,
                    "company_name": "IBM",
                    "raw_jd_text": card.get_text(" ", strip=True) if card else "",
                    "location_city": loc.split(",")[0].strip(),
                    "industry": "Technology / IT Services",
                    "date_posted": datetime.now().strftime("%Y-%m-%d"),
                    "is_active": True,
                    "job_url": href if href.startswith("http") else f"https://www.ibm.com{href}" if href else "",
                    "business_unit": "",
                    "source_platform": "IBM Selenium fallback",
                })
    except Exception as e:
        print(f"  Selenium error: {e}")
    finally:
        driver.quit()

print(f"Total IBM India jobs: {len(ibm_jobs)}")


IBM INDIA JOB SCRAPER
Source: Google Cloud Talent Solution API (jobsapi-google.m-cloud.io)
  Using Google Cloud Talent Solution API...


  [ERROR] HTTP 404
  Trying alternate GET endpoint...


  Total matching jobs: 0

  API approach returned few results. Trying Selenium on ibm.com/careers...


Total IBM India jobs: 0


In [5]:
df_ibm = save_results(ibm_jobs, "IBM", OUTPUT_DIR)
if df_ibm is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_ibm.columns]
    print(df_ibm[cols].head(10).to_string())


  [WARN] No jobs found for IBM
